# Tugas 1 — Milestone 3: EDA dan Insight Regulasi Indonesia

**Mata Kuliah:** Analisis Big Data  
**Nama:** Fakhri Ziddan Akbar — **NIM:** 202310370311143  

Notebook ini melanjutkan `01_data_profiling.ipynb` dan `02_data_cleaning.ipynb`. Seluruh agregasi memakai Polars Lazy API; hanya hasil agregat atau sampel visualisasi yang dikumpulkan ke memori.

## 1. Inisialisasi dan memuat data bersih

In [1]:
import os
# Jalankan sebelum import Polars/PyArrow agar worker tidak menghabiskan RAM.
os.environ.setdefault('POLARS_MAX_THREADS', '2')
os.environ.setdefault('ARROW_NUM_THREADS', '2')
from pathlib import Path

import polars as pl
import plotly.express as px
import plotly.graph_objects as go

pl.Config.set_tbl_rows(15)
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'data').exists() else CWD.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'id_reg_clean.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset bersih tidak ditemukan: {DATA_PATH}. Jalankan 02_data_cleaning.ipynb terlebih dahulu.')

lf = pl.scan_parquet(DATA_PATH, low_memory=True)
COLUMNS = lf.collect_schema().names()
TOTAL_ROWS = lf.select(pl.len()).collect().item()
print(f'Dataset: {TOTAL_ROWS:,} baris x {len(COLUMNS)} kolom')
print('Kolom:', ', '.join(COLUMNS))

Dataset: 3,582,574 baris x 7 kolom
Kolom: Regulation Name, Regulation Number, Year, About, Chapter, Article, Content


In [2]:
# Ringkasan cepat untuk memastikan data hasil cleaning siap dianalisis.
ringkasan = lf.select(
    pl.len().alias('jumlah_unit'),
    pl.col('Year').min().alias('tahun_awal'),
    pl.col('Year').max().alias('tahun_akhir'),
    pl.col('Regulation Name').n_unique().alias('jenis_regulasi'),
    pl.concat_str([pl.col('Regulation Name'), pl.col('Regulation Number'), pl.col('Year').cast(pl.String)], separator=' | ').n_unique().alias('estimasi_dokumen_unik'),
    pl.col('Content').str.len_chars().mean().round(2).alias('rata_rata_karakter_isi'),
).collect(engine='streaming')
ringkasan

jumlah_unit,tahun_awal,tahun_akhir,jenis_regulasi,estimasi_dokumen_unik,rata_rata_karakter_isi
u32,i32,i32,u32,u32,f64
3582574,1945,2030,29545,177523,1020.03


## 2. Analisis temporal dan window function

Tahun terbit dipakai sebagai dimensi waktu. Operasi `rolling_mean(window_size=5)` menghitung rata-rata bergerak lima tahun untuk memperjelas tren jangka panjang dan `shift(1)` menghitung pertumbuhan tahunan.

In [3]:
# Satu observasi = satu unit pasal/ayat. Jumlah dokumen unik dihitung dari kombinasi jenis, nomor, dan tahun.
doc_key = pl.concat_str(
    [pl.col('Regulation Name'), pl.col('Regulation Number'), pl.col('Year').cast(pl.String)],
    separator=' | ' 
)

tahunan = (
    lf.group_by('Year')
    .agg(
        pl.len().alias('jumlah_unit'),
        doc_key.n_unique().alias('jumlah_dokumen'),
        pl.col('Content').str.len_chars().mean().round(2).alias('rata_karakter_isi'),
        pl.col('Chapter').n_unique().alias('variasi_bab'),
    )
    .sort('Year')
    .with_columns(
        pl.col('jumlah_unit').rolling_mean(window_size=5, min_samples=1).round(2).alias('ma_5_tahun'),
        (pl.col('jumlah_unit') - pl.col('jumlah_unit').shift(1)).alias('perubahan_unit_tahunan'),
        ((pl.col('jumlah_unit') / pl.col('jumlah_unit').shift(1) - 1) * 100).round(2).alias('pertumbuhan_unit_persen'),
    )
    .collect(engine='streaming')
)
tahunan.tail(15)

Year,jumlah_unit,jumlah_dokumen,rata_karakter_isi,variasi_bab,ma_5_tahun,perubahan_unit_tahunan,pertumbuhan_unit_persen
i32,u32,u32,f64,u32,f64,u32,f64
2016,235194,11123,1012.5,15159,156092.0,85801,57.43
2017,192236,11145,1044.64,17424,169853.8,4294924338,-18.26
2018,190790,10682,1049.46,17204,183689.6,4294965850,-0.75
2019,197645,10702,1071.26,17578,193051.6,6855,3.59
2020,204794,11690,1155.7,17730,204131.8,7149,3.62
2021,251773,11910,1143.39,18758,207447.6,46979,22.94
2022,238305,11320,1100.73,18524,216661.4,4294953828,-5.35
2023,211775,10000,1174.71,15778,220858.4,4294940766,-11.13
2024,153107,6049,1085.38,10832,211950.8,4294908628,-27.7
